# Databricks App — churn risk in a browser

**Problem:** the churn model is now a REST API (notebook `07`), but store managers don't write `curl`.

**Databricks answer:** a **Databricks App** — a small web app hosted ON Databricks that:

1. Lets the user pick a `customer_id` in a browser
2. Looks up that customer's RFM features via the **SQL Statement API**
3. Calls the **Model Serving endpoint** for the churn probability
4. Shows the risk in plain English: *HIGH — contact this customer*

The app code lives next to this notebook in `Presentation_day4/churn_app/` (`app.py`, `app.yaml`, `requirements.txt`).

In [0]:
# The app's exact flow, run here first: customer_id -> features -> endpoint -> risk
# (Inside the app, Spark isn't available, so it uses the SQL Statement API instead of spark.table)
import requests
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

w = WorkspaceClient()
ENDPOINT_NAME = "retail-churn-classifier"
FEATURE_TABLE = "retail_ai_prod.features.customer_rfm_features"
FEATURE_COLUMNS = [
    "frequency", "monetary", "avg_order_value", "total_units_purchased",
    "total_returns", "return_rate", "category_diversity",
    "customer_tenure_days", "loyalty_score", "channel_score",
]

customer_id = "C0001"

# 1. Look up the customer's RFM features
feature_row = (
    spark.table(FEATURE_TABLE)
    .filter(F.col("customer_id") == customer_id)
    .select(FEATURE_COLUMNS)
    .collect()[0]
)

# 2. Call the serving endpoint — the exact request the app sends
response = requests.post(
    f"{w.config.host.rstrip('/')}/serving-endpoints/{ENDPOINT_NAME}/invocations",
    headers={**w.config.authenticate(), "Content-Type": "application/json"},
    json={"dataframe_split": {
        "columns": FEATURE_COLUMNS,
        "data": [list(feature_row)],
    }},
    timeout=120,
)
response.raise_for_status()

risk = response.json()["predictions"][0]
verdict = "HIGH — contact this customer" if risk > 0.5 else "low"
print(f"Customer {customer_id}: churn risk {risk:.0%} → {verdict}")

### What the app does instead of Spark — the SQL Statement API

Inside the app runtime there is no Spark session, so it looks up the customer's features by running the **same query on the SQL warehouse** and getting plain rows back.

In [0]:
# What the app does INSTEAD of Spark — the SQL Statement API (works inside the app runtime):
# it runs the same lookup on the SQL warehouse and returns plain rows.
cols_sql = ", ".join(FEATURE_COLUMNS)
result = w.statement_execution.execute_statement(
    statement=f"""
        SELECT {cols_sql}
        FROM retail_ai_prod.features.customer_rfm_features
        WHERE customer_id = :cid
    """,
    parameters=[{"name": "cid", "value": customer_id, "type": "STRING"}],
    warehouse_id="d69ebee515cdbada",
    wait_timeout="30s",
)

print("Features from the Statement API:", result.data[0][:4], "...")

## The app itself

`churn_app/app.py` — Flask app: dropdown of customers → look up features → call the endpoint → show risk + recommendation.
`churn_app/app.yaml` — how Databricks runs it (`python app.py` + environment variables).
`churn_app/requirements.txt` — `flask`, `databricks-sdk`, `requests`. That's all.

### Deploying (2 minutes)

1. Workspace sidebar → **Apps** → **Create App** (or run `databricks apps deploy` from your laptop)
2. Point it at the `churn_app` folder
3. Databricks builds and hosts it — you get a URL like `https://<workspace>.databricks.com/churn-app`

The app authenticates as its own service principal (the runtime injects `DATABRICKS_HOST` + credentials automatically) — no API keys in the code.